# Lab 03, Part A: Text to Numbers (Bag-of-Words)

**Course:** Introduction to Artificial Intelligence (AI-101L)

Goal: turn text into a numerical matrix that a machine learning model can use.
**Run it all at once:** Runtime -> Run all.

In [1]:
!pip install -q scikit-learn pandas numpy
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

## Task 3.1: Vectorising text with `CountVectorizer`

In [2]:
# 1. Sample text data (our "corpus")
documents = [
    "The sun is shining today.",
    "The weather is good, the sun is great.",
    "A sunny day is a wonderful day."
]

print("--- Raw Text Documents ---")
for i, doc in enumerate(documents):
    print(f"Doc {i+1}: {doc}")

--- Raw Text Documents ---
Doc 1: The sun is shining today.
Doc 2: The weather is good, the sun is great.
Doc 3: A sunny day is a wonderful day.


In [3]:
# 2. CountVectorizer does tokenization AND vocabulary building in one step.
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(documents)

feature_names = vectorizer.get_feature_names_out()
text_matrix = X_text.toarray()

print("--- Bag-of-Words (BoW) Transformation ---")
print(f"Vocabulary (Features): {feature_names}")
print(f"Shape of Feature Matrix: {text_matrix.shape}   (documents x unique terms)")

--- Bag-of-Words (BoW) Transformation ---
Vocabulary (Features): ['day' 'good' 'great' 'is' 'shining' 'sun' 'sunny' 'the' 'today' 'weather'
 'wonderful']
Shape of Feature Matrix: (3, 11)   (documents x unique terms)


In [4]:
df_text = pd.DataFrame(text_matrix, columns=feature_names,
                       index=[f"Doc {i+1}" for i in range(len(documents))])
print("BoW Numerical Feature Matrix (Machine Understandable Format):\n")
display(df_text)

BoW Numerical Feature Matrix (Machine Understandable Format):



,day,good,great,is,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,1,1,1,0,1,1,0,0
Doc 2,0,1,1,2,0,1,0,2,0,1,0
Doc 3,2,0,0,1,0,0,1,0,0,0,1


In [5]:
# The two "silent decisions" of CountVectorizer, proven rather than trusted.
print("Is 'a' in the vocabulary? ", 'a' in feature_names)
print("Is 'the' in the vocabulary?", 'the' in feature_names)
print()
print("Default token pattern :", vectorizer.token_pattern)
print("Lowercase enabled     :", vectorizer.lowercase)
print()
print("The token pattern needs 2+ word characters, which is why 'a' is dropped.")
print("Lowercase is on, so 'The' and 'the' count as one feature.")

Is 'a' in the vocabulary?  False
Is 'the' in the vocabulary? True

Default token pattern : (?u)\b\w\w+\b
Lowercase enabled     : True

The token pattern needs 2+ word characters, which is why 'a' is dropped.
Lowercase is on, so 'The' and 'the' count as one feature.


In [6]:
# Sparsity: how much of the matrix is zero?
total_cells = text_matrix.size
zero_cells = (text_matrix == 0).sum()
print(f"Matrix cells : {total_cells}")
print(f"Zero cells   : {zero_cells}")
print(f"Sparsity     : {zero_cells / total_cells:.1%}")
print("Type returned by fit_transform:", type(X_text))

Matrix cells : 33
Zero cells   : 18
Sparsity     : 54.5%
Type returned by fit_transform: <class 'scipy.sparse._csr.csr_matrix'>


## In-lab exercises (text)

In [7]:
# --- Exercise 1: unigrams + bigrams ---
vec_1gram = CountVectorizer(ngram_range=(1, 1))
vec_2gram = CountVectorizer(ngram_range=(1, 2))
X1 = vec_1gram.fit_transform(documents)
X2 = vec_2gram.fit_transform(documents)

print(f"ngram_range=(1,1) -> {X1.shape[1]:3d} features")
print(f"ngram_range=(1,2) -> {X2.shape[1]:3d} features")
print(f"Growth factor     : {X2.shape[1] / X1.shape[1]:.1f}x")
print()
print("New bigram features introduced:")
print([f for f in vec_2gram.get_feature_names_out() if ' ' in f])

ngram_range=(1,1) ->  11 features
ngram_range=(1,2) ->  24 features
Growth factor     : 2.2x

New bigram features introduced:
['day is', 'good the', 'is good', 'is great', 'is shining', 'is wonderful', 'shining today', 'sun is', 'sunny day', 'the sun', 'the weather', 'weather is', 'wonderful day']


In [8]:
# --- Exercise 2: a 4th document that repeats "sun" ---
documents_4 = documents + ["sun sun sun sun sun and more sun today"]
vec = CountVectorizer()
M = vec.fit_transform(documents_4).toarray()
df4 = pd.DataFrame(M, columns=vec.get_feature_names_out(),
                   index=[f"Doc {i+1}" for i in range(len(documents_4))])
display(df4)

print("Total token count per document (row sums):")
print(df4.sum(axis=1).to_string())
print()
print("Doc 4 dominates the 'sun' column purely by repetition (count =", df4.loc['Doc 4', 'sun'], ").")
print("Raw counts grow with document length. TF-IDF (Lab 02) corrects this.")

,and,day,good,great,is,more,shining,sun,sunny,the,today,weather,wonderful
Doc 1,0,0,0,0,1,0,1,1,0,1,1,0,0
Doc 2,0,0,1,1,2,0,0,1,0,2,0,1,0
Doc 3,0,2,0,0,1,0,0,0,1,0,0,0,1
Doc 4,1,0,0,0,0,1,0,6,0,0,1,0,0


Total token count per document (row sums):
Doc 1    5
Doc 2    8
Doc 3    5
Doc 4    9

Doc 4 dominates the 'sun' column purely by repetition (count = 6 ).
Raw counts grow with document length. TF-IDF (Lab 02) corrects this.


## Discussion questions (Part A): my answers

**1. What does each column in the matrix represent?**
Each column is one **unique word** (a "term") from the vocabulary of the whole corpus. The number in a column is **how many times that word appears in that document**. In this lab there are 11 columns, so each document becomes a vector of 11 numbers.

**2. What is the machine-understandable format?**
It is the **document-term matrix**: a table of numbers with one row per document and one column per word (here 3 x 11). Every document, whatever its length, becomes a **fixed-length vector of counts**. A model can do maths on numbers, but not on raw sentences.

**3. How is this an oversimplification of the text?**
Information that is lost:
- **Word order** is gone: "dog bites man" and "man bites dog" give the same vector.
- **Grammar and meaning** are gone: the model does not know that "sun" and "sunny" are related.
- **Capital letters** are removed (The = the) and **one-letter words** such as "a" and **punctuation** are dropped.
- All words count equally, so common words like "is" and "the" can dominate.
- The matrix is mostly zeros (54.5% here, and more than 99% for a real corpus), so it is wasteful.

**Exercise 1 (bigrams):** with `ngram_range=(1,2)` the vocabulary grows from 11 to **24** features (2.2x). Every pair of neighbouring words becomes a new feature, so the vocabulary grows quickly with a larger corpus. In return, some word order is kept.

**Exercise 2 (4th document):** the new document adds 2 new columns (`and`, `more`) and puts **6** in the `sun` column (the biggest value in the matrix). Its row sum is 9, bigger than the others (5, 8, 5). Raw counts rise with document length and repetition, so long documents look "stronger" even if they are not more relevant. TF-IDF or normalising the counts fixes this.